# 1. Carga y exploración inicial de los datos
Carga el archivo CSV con la matriz de datos urbanos y espaciales de Bogotá, mostrando una vista previa de su estructura.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [3]:
print("--- 1. CARGA Y DESCRIPCIÓN EXPLORATORIA DE DATOS ---")

# Ruta del archivo unificado en el proyecto
matriz="../data/matriz_barrios_unificada.csv"
df= pd.read_csv(matriz)
display(df.head())

--- 1. CARGA Y DESCRIPCIÓN EXPLORATORIA DE DATOS ---


,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry
0,USME,EL BOSQUE,290000.0,650000.0,660000.0,0.084056,0,0.0,POLYGON ((-74.10014659299998 4.505445796000060...
1,SUBA,BILBAO,750000.0,1790000.0,2050000.0,0.090886,0,0.0,"POLYGON ((-74.113150507 4.751871279999989, -74..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,POLYGON ((-74.15600688799998 4.616521243000022...
3,CHAPINERO,CONCEPCION NORTE,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,POLYGON ((-74.06568096400002 4.656363177999992...
4,PUENTE ARANDA,COLON,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,POLYGON ((-74.11963554699997 4.619638797999983...


In [4]:
def auditar_datos_inmobiliarios(df, etapa="ANTES DE LA IMPUTACIÓN"):
    """
    Función para auditar el estado de los valores de referencia y cagr.
    Muestra conteo de nulos, registros afectados y desglose específico de anomalías.
    """
    print(f"\n================ CLINICAL AUDIT: {etapa} ================")
    
    cols_eval = ['v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25']
    
    # 1. Conteo total de valores nulos (NaN) actuales por columna
    print("\n1. Conteo de valores nulos (NaN) por columna:")
    print(df[cols_eval].isnull().sum())
    
    # 2. Conteo de ceros (0) que representan datos faltantes disfrazados
    print("\n2. Conteo de valores en cero (0) por columna:")
    print((df[cols_eval] == 0).sum())
    
    # 3. Conteo específico por año de anomalías (NaN o 0)
    print("\n3. Desglose detallado de anomalías (NaN o 0) por año:")
    for col in cols_eval:
        n_nans = df[col].isnull().sum()
        n_zeros = (df[col] == 0).sum()
        print(f"   - {col}: {n_nans} NaNs | {n_zeros} ceros (Total anomalías: {n_nans + n_zeros})")
    
    # 4. Filtrar filas con anomalías en el rango clave del CAGR (2015 a 2025) y 2026
    condicion_critica = (
        df['v_ref_2015'].isnull() | (df['v_ref_2015'] == 0) |
        df['v_ref_2025'].isnull() | (df['v_ref_2025'] == 0) |
        df['v_ref_2026'].isnull() | (df['v_ref_2026'] == 0) |
        df['cagr_15_25'].isnull() | (df['cagr_15_25'] == 0) 
    )
    filas_criticas = df[condicion_critica]
    print(f"\n4. Total de registros afectados con anomalías entre 2015 - 2025, 2026 y cagr: {len(filas_criticas)}")
    
    # 5. Mostrar una muestra de los registros afectados si los hay
    if len(filas_criticas) > 0:
        print("\n5. Muestra de registros con anomalías críticas:")
        display(filas_criticas[['localidad', 'barrio', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25']].head(10))
    else:
        print("\n5. ¡No se encontraron registros críticos!")
    print("================================================================")

In [5]:
auditar_datos_inmobiliarios(df, etapa="ESTADO INICIAL")


================ CLINICAL AUDIT: ESTADO INICIAL ================

1. Conteo de valores nulos (NaN) por columna:
v_ref_2015    0
v_ref_2025    0
v_ref_2026    8
cagr_15_25    0
dtype: int64

2. Conteo de valores en cero (0) por columna:
v_ref_2015    0
v_ref_2025    0
v_ref_2026    0
cagr_15_25    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2025: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 8 NaNs | 0 ceros (Total anomalías: 8)
   - cagr_15_25: 0 NaNs | 0 ceros (Total anomalías: 0)

4. Total de registros afectados con anomalías entre 2015 - 2025, 2026 y cagr: 8

5. Muestra de registros con anomalías críticas:


,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25
84,FONTIBON,EL PANTANO,92344.5,34450.00,NaN,-0.093896
166,TEUSAQUILLO,PARQUE DISTRITAL SALITRE,80000.0,139363.69,NaN,0.057075
505,CIUDAD BOLIVAR,EL MOCHUELO III RURAL,27996.0,55498.50,NaN,0.070825
570,KENNEDY,OSORIO XII RURAL,75000.0,111000.00,NaN,0.039983
625,USME,EL MOCHUELO ALTO RURAL,27996.0,69364.00,NaN,0.094974
702,KENNEDY,OSORIO,75000.0,111000.00,NaN,0.039983
873,ENGATIVA,CAMPO EUCARISTICO,80000.0,170930.00,NaN,0.078879
986,USAQUEN,MIRADOR DEL NORTE,80000.0,138000.00,NaN,0.056036


# 2. Auditoría y separación inicial del dataset
Separa el conjunto de datos principal en datos supervisados (que tienen el valor de referencia de 2026 real) y el set de inferencia (los registros huérfanos sin 2026), conservando intactos los nombres de los barrios y las geometrías.

In [6]:
df_supervised = df[df['v_ref_2026'].notnull()].copy()
df_inference = df[df['v_ref_2026'].isnull()].copy()

print(f"Registros para entrenamiento/prueba: {len(df_supervised)}")
print(f"Registros huérfanos para predecir (Inferencia): {len(df_inference)}")

Registros para entrenamiento/prueba: 1090
Registros huérfanos para predecir (Inferencia): 8


# 3. Preparación de la matriz de características (X) y definición de variables categóricas
Prepara la matriz de entrenamiento manteniendo barrio y localidad, excluyendo únicamente la geometría, el año 2015 y el target de 2026, y asegurándose de que las variables categóricas estén en formato texto para CatBoost.

In [7]:
# Vector Target (lo que queremos predecir)
y_sup = df_supervised['v_ref_2026']

# Columnas a excluir para CatBoost (MANTENEMOS barrio y localidad vivas)
cols_a_excluir = ['geometry', 'v_ref_2015', 'v_ref_2026']
X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])

# Definir explícitamente cuáles son las columnas categóricas para CatBoost
cat_features = ['localidad', 'barrio']

# Asegurar que las variables categóricas estén en formato texto (string)
for col in cat_features:
    if col in X_sup.columns:
        X_sup[col] = X_sup[col].astype(str)

display(X_sup.head())

,localidad,barrio,v_ref_2025,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb
0,USME,EL BOSQUE,650000.0,0.084056,0,0.0
1,SUBA,BILBAO,1790000.0,0.090886,0,0.0
2,KENNEDY,CIUDAD KENNEDY CENTRAL,2900000.0,0.112346,0,7.0
3,CHAPINERO,CONCEPCION NORTE,3580000.0,0.098438,1,0.0
4,PUENTE ARANDA,COLON,2850000.0,0.081658,0,16.0


In [8]:
df_supervised

,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry
0,USME,EL BOSQUE,290000.0,650000.0,660000.0,0.084056,0,0.0,POLYGON ((-74.10014659299998 4.505445796000060...
1,SUBA,BILBAO,750000.0,1790000.0,2050000.0,0.090886,0,0.0,"POLYGON ((-74.113150507 4.751871279999989, -74..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,POLYGON ((-74.15600688799998 4.616521243000022...
3,CHAPINERO,CONCEPCION NORTE,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,POLYGON ((-74.06568096400002 4.656363177999992...
4,PUENTE ARANDA,COLON,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,POLYGON ((-74.11963554699997 4.619638797999983...
...,...,...,...,...,...,...,...,...,...
1093,CIUDAD BOLIVAR,MEISSEN,64000.0,1198158.0,1395000.0,0.340397,0,1.0,POLYGON ((-74.13462513299999 4.560979234999991...
1094,RAFAEL URIBE URIBE,LOS MOLINOS,450000.0,2750000.0,2600000.0,0.198428,0,0.0,POLYGON ((-74.10752949900001 4.558032486000002...
1095,SUBA,EL PLAN,1400000.0,3400000.0,3798483.0,0.092786,0,38.0,POLYGON ((-74.06395278399998 4.741811476000009...
1096,CIUDAD BOLIVAR,ISMAEL PERDOMO,400000.0,1040000.0,1050000.0,0.100265,0,0.0,POLYGON ((-74.16717365199997 4.581548365999993...


# 4. División de datos en conjuntos de entrenamiento y prueba
Divide la matriz supervisada en conjuntos de entrenamiento (80%) y prueba (20%) para evaluar de manera objetiva el rendimiento del modelo.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)
print(f"Tamaño X_train: {X_train.shape}, X_test: {X_test.shape}")

Tamaño X_train: (872, 6), X_test: (218, 6)


# 5. Entrenamiento y evaluación del modelo CatBoost
Inicializa y entrena el algoritmo CatBoost Regressor pasando la lista de cat_features para que gestione de forma nativa los barrios y localidades, calculando las métricas de desempeño ($R^2$, RMSE, MAE y MSE).

In [10]:
# Inicializar y entrenar CatBoost con soporte nativo de categorías
cat_model = CatBoostRegressor(
    iterations=600,          # Número de árboles
    learning_rate=0.04,      # Tasa de aprendizaje suave
    depth=6,                 # Profundidad de los árboles
    loss_function='RMSE',    # Métrica objetivo
    random_seed=42,
    verbose=100              # Muestra el progreso cada 100 iteraciones
)

# Entrenar pasando la lista de cat_features
cat_model.fit(X_train, y_train, cat_features=cat_features)

# Predecir sobre el set de prueba
y_pred_test = cat_model.predict(X_test)

# Calcular métricas de desempeño
mse_cat = mean_squared_error(y_test, y_pred_test)
rmse_cat = np.sqrt(mse_cat)
mae_cat = mean_absolute_error(y_test, y_pred_test)
r2_cat = r2_score(y_test, y_pred_test)

print("--- RESULTADOS CATBOOST (CON BARRIO Y LOCALIDAD NATIVOS) ---")
print(f"MSE:  {mse_cat:,.2f}")
print(f"RMSE: {rmse_cat:,.2f}")
print(f"MAE:  {mae_cat:,.2f}")
print(f"R2:   {r2_cat:.4f}")

0:	learn: 1737359.2354930	total: 222ms	remaining: 2m 13s
100:	learn: 453440.8901737	total: 8.14s	remaining: 40.2s
200:	learn: 379571.8045515	total: 16.3s	remaining: 32.3s
300:	learn: 342107.3394302	total: 26.2s	remaining: 26s
400:	learn: 310577.0331057	total: 34.2s	remaining: 17s
500:	learn: 287052.8257109	total: 42s	remaining: 8.29s
599:	learn: 266860.8590678	total: 49.9s	remaining: 0us
--- RESULTADOS CATBOOST (CON BARRIO Y LOCALIDAD NATIVOS) ---
MSE:  154,442,897,249.83
RMSE: 392,992.24
MAE:  235,324.50
R2:   0.9449


# 6. Inferencia, consolidación y formato final para Dash y la API
Realiza la predicción manteniendo el formato de texto en las categorías para los registros huérfanos y supervisados, consolida el dataset maestro, calcula la hipervalorización y ajusta los formatos numéricos.

In [11]:
# 1. Preparar features de inferencia y asegurar formato string en categorías
X_inf = df_inference.drop(columns=[col for col in cols_a_excluir if col in df_inference.columns])
for col in cat_features:
    if col in X_inf.columns:
        X_inf[col] = X_inf[col].astype(str)

# Predicción para los 8 registros huérfanos
df_inference['v_ref_2026_predicho'] = cat_model.predict(X_inf)

# 2. Generar predicción para todo el set supervisado
X_sup_completo = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])
for col in cat_features:
    if col in X_sup_completo.columns:
        X_sup_completo[col] = X_sup_completo[col].astype(str)

df_supervised['v_ref_2026_predicho'] = cat_model.predict(X_sup_completo)

# 3. Unificar todo en un solo DataFrame maestro consolidado (con barrios y geometría intactos)
df_final_maestro = pd.concat([df_supervised, df_inference], ignore_index=True)

# 4. Calcular métricas clave para el Dashboard en Dash
# Porcentaje de hipervalorización proyectada usando el valor predicho
df_final_maestro['hipervalorizacion_pct'] = ((df_final_maestro['v_ref_2026_predicho'] - df_final_maestro['v_ref_2025']) / df_final_maestro['v_ref_2025']) * 100

# Clasificación de los datos para filtros en Dash
df_final_maestro['tipo_registro'] = df_final_maestro['v_ref_2026'].apply(lambda x: 'Histórico Oficial' if pd.notnull(x) else 'Imputado / Predicho (Huérfano)')

# 5. Ajustar el precio predicho a número normal (redondeado a 0 decimales ya que son pesos en COP)
df_final_maestro['v_ref_2026_predicho'] = df_final_maestro['v_ref_2026_predicho'].round(0)

# 6. Ajustar la hipervalorización en porcentaje estrictamente a 2 decimales
df_final_maestro['hipervalorizacion_pct'] = df_final_maestro['hipervalorizacion_pct'].round(2)

# 7. Desactivar la notación científica globalmente en Pandas para que se vea bien en el notebook
pd.set_option('display.float_format', lambda x: '%.2f' % x)

display(df_final_maestro.head())

,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry,v_ref_2026_predicho,hipervalorizacion_pct,tipo_registro
0,USME,EL BOSQUE,290000.00,650000.00,660000.00,0.08,0,0.00,POLYGON ((-74.10014659299998 4.505445796000060...,650554.00,0.09,Histórico Oficial
1,SUBA,BILBAO,750000.00,1790000.00,2050000.00,0.09,0,0.00,"POLYGON ((-74.113150507 4.751871279999989, -74...",2184065.00,22.01,Histórico Oficial
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.00,2900000.00,2640000.00,0.11,0,7.00,POLYGON ((-74.15600688799998 4.616521243000022...,3146491.00,8.50,Histórico Oficial
3,CHAPINERO,CONCEPCION NORTE,1400000.00,3580000.00,4380000.00,0.10,1,0.00,POLYGON ((-74.06568096400002 4.656363177999992...,4163153.00,16.29,Histórico Oficial
4,PUENTE ARANDA,COLON,1300000.00,2850000.00,3170000.00,0.08,0,16.00,POLYGON ((-74.11963554699997 4.619638797999983...,3265796.00,14.59,Histórico Oficial


# 7. Exportación de archivos para Dash y consumo por API
Exporta el resultado final en formato GeoPackage (.gpkg) con sus respectivas geometrías de polígonos y atributos de barrio para visualizar en Dash, y en formato Parquet para la API.

In [12]:
# 8. Exportación de archivos finales adaptados para Dash y API
# A. Convertir el DataFrame maestro en un GeoDataFrame real usando los polígonos
#gdf_export = gpd.GeoDataFrame(df_final_maestro, geometry=gpd.GeoSeries.from_wkt(df_final_maestro['geometry']), crs="EPSG:4326")

# B. Exportar en formato .gpkg (GeoPackage) - Recomendado para Dash y mapas espaciales
#gdf_export.to_file("output_hipervalorizacion_bogota_2026_catboost.gpkg", driver="GPKG")
#print("\n¡Archivo geoespacial en .gpkg exportado con éxito para tu dashboard en Dash!")

# C. Exportar versión tabular sin geometría por si tu API la requiere para consultas rápidas
#df_final_maestro.drop(columns=['geometry']).to_parquet("datos_api_bogota_catboost.parquet", index=False)
#print("¡Archivo tabular para la API exportado con éxito: 'datos_api_bogota_catboost.parquet'!")